In [ ]:
# /// script
# requires-python = ">=3.11,<3.14"
# dependencies = [
#     "azure-identity>=1.19.0",
#     "azure-mgmt-cognitiveservices>=13.5.0",
#     "azure-ai-projects>=2.0.0",
#     "azure-search-documents>=11.5.0",
#     "python-dotenv>=1.0.0",
# ]
# ///
# Launch with one command (uv builds the environment + kernel for you):
#     uvx juv run m03_ops_tour.ipynb
# Prereqs: `az login`, and a .env in this folder (see .env.example).
print("Ready. Everything below is keyless and safe to re-run.")

# Lesson 3 — Managing and monitoring AI in Foundry (hands-on tour)

Once a Microsoft Foundry solution is live, three questions decide whether it stays healthy:
**do I have capacity, is the model behaving, and is my search data trustworthy?**
This notebook answers all three against a real project, reading **live** signals.

Two things make it safe to lean on:
- **Keyless.** Everything authenticates with your `az login` identity (Microsoft Entra ID) — no API keys anywhere.
- **Idempotent.** Every cell reads or converges to the same state, so you can run the whole thing top to bottom as many times as you like.

You'll also see each of these in the portal; the notebook is the same truth, in code you could automate.

In [ ]:
# --- Setup: one credential, read config from .env -------------------------------------
import os
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()  # reads the .env in this folder
cred = AzureCliCredential()  # your `az login` identity

SUBSCRIPTION = os.environ["AZURE_SUBSCRIPTION_ID"]
REGION       = os.environ.get("QUOTA_LOCATION", "eastus2")
PROJECT_EP   = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
SEARCH_EP    = os.environ.get("SEARCH_ENDPOINT", "")
print("project :", PROJECT_EP)
print("region  :", REGION, "| search:", SEARCH_EP or "(not set yet)")

## 1 · Capacity — quota and headroom

Every model deployment draws on a **quota** measured in **tokens per minute (TPM)**, granted
per subscription, per region, per model. Quota is the invisible wall your architecture runs into.

When a deployment creeps toward its limit, that's the cue to **scale** — raise TPM, add PTUs, or
spread traffic across regions — *before* users start seeing throttling (HTTP 429). The number that
matters isn't how much you're using, it's how much **headroom** is left.

A useful way to hold it: on the exam, "a new deployment fails because there's no capacity in the
region" is a **quota** problem, not a code problem. The fix is to free or request TPM in that region.

In [ ]:
# Read the model quota for this region -- current usage vs the granted limit (headroom).
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
arm = CognitiveServicesManagementClient(cred, SUBSCRIPTION)

print(f"{'quota':<44}{'used':>8}{'limit':>8}{'headroom':>10}")
print("-" * 70)
for u in arm.usages.list(REGION):
    limit = u.limit or 0
    if limit <= 0:
        continue
    if "gpt-5" not in (u.name.value or "").lower():
        continue  # keep the on-screen list focused on the models this course uses
    used = u.current_value or 0
    name = (u.name.value or "").replace("OpenAI.", "")
    print(f"{name:<44}{used:>8.0f}{limit:>8.0f}{limit-used:>10.0f}")
print("\nRule of thumb: when headroom gets thin, scale or request quota before users feel it.")

## 2 · Behavior — the observability pipe

Foundry **observability** is three capabilities working together:
- **Evaluation** — quality and safety scores (coherence, groundedness, content safety).
- **Monitoring** — latency, token use, and error rates on a live dashboard.
- **Tracing** — the step-by-step execution of a call or an agent.

All of it flows through **Azure Monitor Application Insights**. So the very first health check is
simply: *is that pipe connected?* No connection, no signals — which is itself worth knowing, because
a "monitoring shows nothing" problem is almost always a missing or mis-wired Application Insights link.

In [ ]:
# Confirm the observability pipe: fetch the project's Application Insights connection.
from azure.ai.projects import AIProjectClient
from azure.core.exceptions import ResourceNotFoundError

project = AIProjectClient(endpoint=PROJECT_EP, credential=cred)
try:
    conn = project.telemetry.get_application_insights_connection_string()
    print("Application Insights: CONNECTED")
    print("  ", conn.split(";")[0], "...")   # show only the InstrumentationKey portion
    print("This pipe carries latency, token, safety, and groundedness signals into the dashboard.")
except ResourceNotFoundError:
    print("Application Insights: NOT connected for this project.")
    print("Connect one under the project's Monitoring settings to light up production monitoring.")

## 3 · Data — index health and relevance

A retrieval-augmented app is only as good as the **search index** behind it. When an index quietly
breaks — documents drop out, key fields go empty — the answers get worse with **no error at all**.

So you watch two things:
- **Health** — document count and storage (did ingestion do what you expected?).
- **Relevance** — does a real query return sensible results, and how strong is the top **search score**?

Below we stand up a tiny index, add a few documents, then read both signals. Creating the index and
adding the documents are both *upserts*, so re-running this cell just converges — nothing duplicates.

In [ ]:
# Idempotently ensure a small demo index exists and has a few documents.
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchFieldDataType,
)
from azure.search.documents import SearchClient

if not SEARCH_EP:
    print("SEARCH_ENDPOINT not set yet -- skip this cell until the search service is wired into .env.")
else:
    index_name = "ai103-health-demo"
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="title", type=SearchFieldDataType.String),
        SearchableField(name="content", type=SearchFieldDataType.String),
    ]
    index_client = SearchIndexClient(SEARCH_EP, cred)
    index_client.create_or_update_index(SearchIndex(name=index_name, fields=fields))  # upsert = idempotent

    docs = [
        {"id": "1", "title": "Quota", "content": "Tokens per minute govern a deployment's throughput."},
        {"id": "2", "title": "Observability", "content": "Application Insights carries latency and groundedness signals."},
        {"id": "3", "title": "Index health", "content": "Document count and relevance scores reveal a broken index."},
    ]
    SearchClient(SEARCH_EP, index_name, cred).merge_or_upload_documents(docs)  # upsert = idempotent
    print(f"Index '{index_name}' ready with {len(docs)} documents (re-runnable).")

In [ ]:
# Read the two signals: HEALTH (doc count, storage) and RELEVANCE (top search score).
if not SEARCH_EP:
    print("SEARCH_ENDPOINT not set yet -- skip until the search service is wired into .env.")
else:
    stats = SearchIndexClient(SEARCH_EP, cred).get_index_statistics("ai103-health-demo")
    print(f"HEALTH    documents: {stats['document_count']}   storage: {stats['storage_size']} bytes")

    results = SearchClient(SEARCH_EP, "ai103-health-demo", cred).search(
        search_text="throughput and latency signals", top=3, include_total_count=True
    )
    print(f"RELEVANCE query hits: {results.get_count()}")
    for r in results:
        print(f"   score {r['@search.score']:.4f}  ->  {r['title']}")
    print("\nA quiet drop in count, or scores trending toward zero, is your early warning.")

## Recap

Three signals, one project:
- **Capacity** — TPM headroom tells you when to scale.
- **Behavior** — the Application Insights pipe carries latency, safety, and groundedness.
- **Data** — index health and relevance scores catch a silently degrading RAG pipeline.

Read them on a cadence and you catch trouble before your users do. Everything here was keyless and
idempotent, so it's the same code you'd drop into a scheduled job.